# Metodolog?a para clasificaci?n
### Aprendizaje Autom?tico - Instituto de Computaci?n - UdelaR


## Metodolog?a para clasificaci?n

- Indepedientemente del m?todo utilizado, existen etapas comunes para poder hacer aprendizaje supervisado (en particular, clasificaci?n).

- ?Cu?l es nuestra tarea?: dado un conjunto $X$ de instancias independientes con una cierta distribuci?n $D$, cada una de ellas con una clase $y$ asociada, queremos construir una funci?n de clasificaci?n que, dada una instancia nueva, nos devuelva su clase.  

- Algunas preguntas: ?c?mo aprendo la funci?n?, ?sobre qu? instancias?, ?c?mo eval?o la performance de la funci?n?

<img src="https://raw.githubusercontent.com/pln-fing-udelar/curso_aa/master/img/clasificador.png" alt="Drawing" width="400"/>


## Fase 1: Preprocesamiento

-  Vamos a suponer que, para poder entrenar un clasificador, debemos partir de un conjunto $D = \{(x_i,y_i)\}$, llamado conjunto de entrenamiento, donde cada instancia $x_i \in \mathbb{R}^n$ y $y_i \in \mathbb{R}$ (no todos los algoritmos de aprendizaje necesitan este formato de entrada, es solamente para fijar ideas)

- (Des) afortunadamente, los conjuntos de datos que generalmente disponemos surgen de sensores, datos ingresados por humanos, fuentes diferentes, etc. Por lo tanto, debemos limpiarlos (_data cleaning_).

- El formato de los datos originales puede ser diverso: elementos de un conjunto (categ?ricos), fechas, textos, im?genes, etc. Debemos buscar formas para convertirlos a un formato aceptable por el algoritmo (_data transformation_).

- Nuestros datos pueden venir de diferentes fuentes, debemos integrarlos (_data integration_)

- Puede que, para ser m?s eficientes en los tiempos de aprendizaje, sea necesario agrupar datos, eliminar atributos o reducir el numero de instancias, buscando no perder infromaci?n (_data reduction_)

![Data Preprocessing](https://miro.medium.com/max/628/1*d1P90NT33rRKlJT7opFO8w.png)

Fuente de la imgen: [Data Preprocessing](https://medium.com/@silicon.smile1/data-preprocessing-b1552b4060f3) - Umar Farooq 

#### Titanic

Trabajaremos con  un ejemplo utilizando pandas (ya que estamos, importamos otras bibliotecas que probablemente utilicemos). 
- Titanic Dataset: listado de pasajeros del Titanic, indicando si sobrevivieron o no. M?s detalles [aqu?](https://www.kaggle.com/c/titanic). 

In [ ]:
import numpy as np
import pandas as pd
import sklearn
import sklearn.preprocessing
import sklearn.feature_selection
pd.options.mode.chained_assignment = 'warn'  # default='warn'
import graphviz


In [ ]:
titanic=pd.read_csv('https://raw.githubusercontent.com/pln-fing-udelar/curso_aa/master/data/titanic.csv')
titanic

### Conjunto de entrenamiento y Testeo. Accuracy (Spoiler)

<img src="https://miro.medium.com/max/1940/1*Nv2NNALuokZEcV6hYEHdGA.png" alt="Drawing" width=400 />


- Separaci?n inicial: conjunto de entrenamiento y de evaluaci?n. 

- Cuantas  m?s instancias para entrenar tengamos, probablemente mejor ser? nuestro modelo, PERO...

- Cuantas m?s intancias para evaluar tengamos, menor ser? la varianza de nuestros resultados.

- Usualmente se divide como 80%-20%, o 70%-30%

- Accuracy: proporci?n de las instancias del conjunto de evaluaci?n que nuestro modelo clasifica correctamente

Fuente de la imagen: [About Train, Validation and Test Sets in Machine Learning](https://towardsdatascience.com/train-validation-and-test-sets-72cb40cba9e7) - Tarang Shah

### Titanic: divisi?n del conjunto de entrenamiento

In [ ]:
# Primero separamos las X de las y. 
titanic_X = titanic.drop(['survived'], axis=1, inplace=False)
titanic_y = titanic[['survived']]

# Construimos los corpus de entrenamiento y test

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(titanic_X, titanic_y, test_size=0.25, random_state=33)

display(X_train.shape)
display(X_test.shape)


### Valores faltantes


- ?qu? hacemos si alg?n atributo de alguna instancia no tiene asignado un valor?

- Opci?n 1: eliminar instancias. Problema: reduce el dataset.

- Opci?n 2: asignar un valor especial (UNK, -1, 0, etc). Esto indica que, si faltan datos, quiere decir algo. 

- Opci?n 3: asignar el valor medio (o la mediana, o la moda) del atributo en el conjunto de entrenamiento

- Opci?n 4: asignar seg?n el m?todo de aprendizaje que estemos utilizando (e.g. lo visto en ?rboles de Decisi?n)

Observaci?n: estos cambios aplican a todo el dataset, pero cualquier c?lculo de estad?stica debe hacerse sobre el conjunto de entrenamiento 

- Algunos detalles en Python: [How to handle missing data with Python](https://machinelearningmastery.com/handle-missing-data-python/) - Jason Brownlee


### Titanic: atributos faltantes

Sustiuimos las edades que nos faltan por el promedio

In [ ]:
X_train

In [ ]:
# Contamos cu?ntos NaN son
print("Cantidad de instancias sin valor: {0}".format(X_train['age'].isnull().sum()))

# Vemos el promedio de edad de los sobrevivientes, seg?n la clase
mean_age=X_train["age"].mean()
display('Valor de mean_age:'+str(mean_age))

# Actualizamos con la mean_age de cada grupo tanto entrenamiento como evaluaci?n
X_train.loc[X_train['age'].isnull(),'age']=mean_age
X_test.loc[X_test['age'].isnull(),'age']=mean_age



In [ ]:
X_train

### Atributos categ?ricos

- Los atributos categ?ricos son atributos cuyos valores pertenecen a un conjunto discreto y finito (y, a veces, no num?rico) 

- Opci?n 1: Cuando tenemos $n$ etiquetas, convertir a valores enteros en el rango $[0.. n-1]$. Problema: sigue siendo discreto, e induce un orden entre las etiquetas. Lo primero puede puede perjudicar a algoritmos que asumen valores continuos, lo segundo puede no representar la realidad.

- Opci?n 2: one-hot-encoding. Creamos tantos atributos nuevos como etiquetas diferentes haya.  En cada instancia, si el valor del atributo original es $i$, el atributo correspondiente al $i$-?simo valor valdr? 1, y el resto valdr?n 0.

### Titanic: codificaci?n de atributos categ?ricos



In [ ]:
# Creamos un labelEncoder utilizando scikit-learn
le=sklearn.preprocessing.LabelEncoder()
# Obtenemos las clases a partir de los valores del conjunto de entrenamiento
le.fit(titanic['sex'])
# Mostramos las clases obtenidas
le.classes_
# Ajustamos el campo sex, transform?ndolo

X_train['sex'] = le.transform(X_train['sex'])
X_test['sex'] = le.transform(X_test['sex'])

In [ ]:
X_train

In [ ]:
# Utilizamos scikit-learn para crear un one-hot-encoder
ohe=sklearn.preprocessing.OneHotEncoder(sparse_output=False)

# Obtenemos las categor?as a partir de los datos de entrenamiento
# Observemos que aqu? utilizo todo el dataset, porque la decisi?n no involucra el aprendizaje
ohe.fit(titanic['pclass'].to_numpy().reshape(-1,1))
display(ohe.categories_)

# Obtenemos los nuevos valores a partir del valor original
new_train=ohe.transform(X_train['pclass'].to_numpy().reshape(-1,1))
new_test =ohe.transform(X_test['pclass'].to_numpy().reshape(-1,1))


# Creamos nuevos atributos
X_train['class_1st']=new_train[:,0]
X_train['class_2nd']=new_train[:,1]
X_train['class_3rd']=new_train[:,2]
X_test['class_1st']=new_test[:,0]
X_test['class_2nd']=new_test[:,1]
X_test['class_3rd']=new_test[:,2]


In [ ]:
X_train

In [ ]:
# Elminamos atributos que no vamos a utilizar para el aprendizaje
# Esto podr?amos afinarlo usando Feature Selection
X_train.drop(['row.names','pclass', 'name', 'embarked', 'home.dest', 'room', 'ticket', 'boat'], axis=1, inplace=True)
X_test.drop(['row.names','pclass', 'name', 'embarked', 'home.dest', 'room', 'ticket', 'boat'], axis=1, inplace=True)

X_train

### Ingenier?a de atributos - Textos

- M?todo tradicional: Bag of Words (BOW): a partir de un vocabulario (lista de palabras del lenguaje), 
construimos un vector con un atributo por palabra. Valores que toma:

    - 1/0: 1 indica que la palabra existe en el texto, 0 que no. 
    - Cantidad de ocurrencias de la palabra en el texto (eventualmente normalizada, dividiendo sobre el total de palabras del documento)
    - tf-idf: pondera la frecuencia de la palabra viendo qu? tan com?n es en general (un valor alto indica que la palabra es com?n en el texto de la instancia, pero rara en el dataset).
    
        - $ tf  = \frac{count}{total}$, siendo $count$ el n?mero de ocurrencias de la palabra en el texto, y $total$ el n?mero total de palabras en el texto
        - $ idf = \log\frac{N}{n}$, siendo $N$ el n?mero de instancias del conjunto, y $n$ el n?mero de instancias donde la palabra aparece en el texto
        - $tf.idf = td \times idf$


### Ingenier?a de atributos - Textos

Ejercicio: supongamos que la palabra "the" aparece en 98 de 100 documentos (instancias), y en mi instancia aparece 9 veces (el documento tiene 200 palabras). An?logamente "computer" aparece 3 veces en mi texto, y en 8 de 100 instancias. Construya los vectores seg?n los criterios descritos. El total de palabras distintas en el corpus es 850, pero solamente vamos a utilizar las 300 m?s comunes (es decir que el vector de cada instancia tendr? 300 elementos). 
- Valor para la posici?n correspondiente a la palabra "the":
    - Si usamos 1/0: 1 (la palabra aparecen en la instancia)
    - Si usamos cant. ocurrencias: 9
    - Si usamos cant. ocurrencias, normalizado: 9/200 = 0.045
    - Si usamos tf-idf: $\frac{9}{200} \cdot \log \frac{100}{98} = 9.09 \times 10^{-4}$

- Valor para la posici?n correspondiente a la palabra "computer":
    - Si usamos 1/0: 1 (la palabra aparecen en la instancia)
    - Si usamos cant. ocurrencias: 3
    - Si usamos cant. ocurrencias, normalizado: 3/200 = 0.015
    - Si usamos tf-idf: $\frac{3}{200} \cdot \log \frac{100}{8} = 0.037$

### Ingenier?a de atributos - Textos

- Antes de crear los vectores, es usual preprocesar el texto: dividir en tokens, eliminar palabras muy comunes (stop words), hacer lematizaci?n (buscar representantes comunes a varias palabras relacionadas). 
- El enfoque BOW no tiene en cuenta el orden de las palabras. Una mejora: en vez de palabras, contar n-gramas (secuencias de n palabras). A este enfoque se lo llama tambi?n bag-of-ngrams.
- Podemos imaginar a una representaci?n de conteo normalizado BOW como una suma de vectores one-hot-encoded, uno por cada palabra, donde en la posici?n de la palabra est? el valor correspondiente al conteo, y el resto de las posiciones son 0.  

$$
x = \frac{1}{|D|}\sum_{i=1}^{|D|} x^{D_{[i]}}
$$



### Ingenier?a de atributos - Textos

- Problemas de las representaciones anteriores: son muy dispersas (en cada instancia hay muchos atributos con valor 0). No hay noci?n de similaridad entre palabras, si nuestra base de representaci?n para una palabra es un one-hot-encoding.
- Hoy pr?cticamente todos los m?todos en el estado del arte utilizan _word embeddings_: se generan vectores densors de baja dimensionalidad (50-200 atributos), y con algunas propiedades interesantes, a partir del contexto en el que aparece cada palabra usualmente. 


Si quieren saber m?s sobre textos y c?mo procesarlos, pueden hacer el curso [Introducci?n al Procesamiento del Lenguaje Natural](https://eva.fing.edu.uy/course/view.php?id=211), o [Redes Neuronales para Lenguaje Natural](https://eva.fing.edu.uy/course/view.php?id=1758) dictado en esta misma instituci?n, por este mismo grupo de investigaci?n.

### Estandarizaci?n de atributos

- Muchos algoritmos en aprendizaje autom?tico (e.g. knn, redes neuronales, PCA) se benefician de que los atributos continuos tengan aproximadamente el mismo orden de magnitud. Esto se debe, por ejemplo, a que se utiliza la distancia euclidiana y se busca que todos los atributos "pesen" igual al calcularla. En el caso de algoritmos que utilizan descenso por gradiente, puede haber mucha diferencia en performance. 

- Min-max scaling: este escalado deja los valores en el rango $[0-1]$. Dado un valor $x$, obtenemos:

    $ x_s = \frac{x - x_{min}}{x_{max} - x_{min}}$
    
    siendo $x_{min}$ y $x_{max}$ los valores m?nimo y m?ximo respectivamente en el dataset
    
- Normalizaci?n: se escalan los atributos para que tengan las propiedades de una distribuci?n normal est?ndar, con $\mu = 0$ y $\sigma = 1$. 

    $ x_{norm} = \frac{x_i - \mu_{i}}{s_i} $
    
    siendo $\mu_{i}$ la media y  $s_i$ la desviaci?n est?ndar de la muestra.
    
    
 

Ejercicio: normalice los siguientes valores utilizando los dos m?todos anteriores: $\{85,35,42,8,15, 22\}$. ?En qu? rango quedan los valores obtenidos por normalizaci?n?

- Valor m?nimo: 8
- Valor m?ximo: 85
- Valores escalados min-max: {1,0.35, 0.44, 0, 0.09, 0.18}



- Media: $\frac{85+35+42+8+15+22}{6} = 34.50 $
- Desviaci?n est?ndar: $\sqrt{(85 -34.5)^2 + \ldots + (22 - 34.5)^2} = 25.32$
- Valores normalizados: $\{ 1.99,  0.02,  0.3,  -1.05, -0.77, -0.49\}$

(Verifique que los nuevos valores tienen media 0 y desviaci?n est?ndar 1)

- Lectura: [About standarization](https://sebastianraschka.com/Articles/2014_about_feature_scaling.html) - Sebastian Rashcka
- Nota: la estandarizaci?n, as? como la selecci?n de los atributos, debe hacerse durante el entrenamiento (no antes). V?ase [este link](https://stats.stackexchange.com/questions/77350/perform-feature-normalization-before-or-within-model-validation) por m?s detalles.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x=[1,2,3,4,5,6]
valores = np.array([85,35,42,8,15,22])

valores_escalados = np.array([1,0.35,0.44,0,0.09,0.18])

valores_estandarizados = np.array([1.99,0.02,0.3,-1.05,-0.77,-0.49])

fig, axs = plt.subplots(3)
axs[0].plot(x, valores)
axs[1].plot(x, valores_escalados)
axs[2].plot(x, valores_estandarizados)



## Fase 2: Divisi?n del Conjunto de Datos

### Conjunto de entrenamiento, testeo, [y validaci?n]


- Debemos asegurar, para evitar el sobreajuste (_overfitting_), que la evaluaci?n del modelo se realice en un conjunto de datos distinto a aquel sobre el cual se entren?. 

- Sobreajuste: nuestro modelo tiene buen rendimiento sobre el dataset de entrenamiento, pero sus resultados son muy inferiores cuando se encuentra a datos no vistos previamente. Causa probable: el modelo est? memorizando los datos de entrenamiento, sin poder generalizar. 

- Si evaluamos sobre el mismo conjunto de datos sobre el que entrenamos, no podemos saber si estamos sobreajustando. **Sobreajustar es el peligro mayor cuando hacemos aprendizaje autom?tico**

### Conjunto de entrenamiento, testeo, [y validaci?n]

<img src="https://www.researchgate.net/publication/339680577/figure/fig2/AS:865364518924290@1583330387982/llustration-of-the-underfitting-overfitting-issue-on-a-simple-regression-case-Data.png" alt="Drawing" width=600 />


[Ejemplo COVID-19 en Uruguay](https://github.com/gmonce/datascience/blob/master/src/Sobreajustando.ipynb)

### Conjunto de entrenamiento, testeo, [y validaci?n]

<img src="https://vitalflux.com/wp-content/uploads/2020/12/overfitting-and-underfitting-wrt-model-error-vs-complexity.png" alt="Drawing" width=400 />



### Conjunto de entrenamiento, testeo, [y validaci?n]

<img src="https://miro.medium.com/max/1940/1*Nv2NNALuokZEcV6hYEHdGA.png" alt="Drawing" width=400 />


- Separaci?n inicial: conjunto de entrenamiento y de evaluci?n. 

- Cuantas  m?s instancias para entrenar tengamos, probablemente mejor ser? nuestro modelo, PERO...

- Cuantas m?s intancias para evaluar tengamos, menor ser? la varianza de nuestros resultados.

- Usualmente se divide como 80%-20%, o 70%-30%

Fuente de la imagen: [About Train, Validation and Test Sets in Machine Learning](https://towardsdatascience.com/train-validation-and-test-sets-72cb40cba9e7) - Tarang Shah

### Conjunto de validaci?n

- Si queremos ajustar los par?metros del modelo (lo veremos en breve), no es conveniente hacerlo en el conjunto de evaluaci?n (ya que podr?amos estar sobreajustando, nuevamente). 

- Opci?n 1: separar una parte del conjunto de entrenamiento para utilizarlo en esa etapa

- Opci?n 2: utilizar validaci?n cruzada (cross-validation)

### Estratificaci?n

- Aunque no queremos que la evaluaci?n se realice en el conjunto de entrenamiento, s? nos interesa que la distribuci?n de los ejemplos en uno y otro sea similar.

- Al hacer la divisi?n de los conjuntos, lo usual es elegir las instancias al azar, para evitar que las agrupaciones u ordenamientos presentes en el conjunto original puedan dar lugar a distribuciones distintas. 

- Un paso m?s, especialmente importante cuando las clases objetivo est?n _desbalanceadas_ (es decir, hay muchos m?s ejemplos de una clase que de otras), es estratificar: elegir las instancias en cada una de las subclases, obligando a que la proporci?n sea la misma en el corpus de entrenamiento y en el de evaluaci?n.

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/f/fa/Stratified_sampling.PNG/440px-Stratified_sampling.PNG" alt="Drawing" width="300" />



### Conjuntos de datos desbalanceados

- Los conjuntos de datos donde el n?mero de instancias de cada clase son muy diferentes, nos traen problemas para el aprendizaje, ya que un m?todo que favorezca a la clase mayoritaria generalmente tendr? "buenos resultados" en t?rminos de _accuracy_

<img src="https://imbalanced-learn.org/stable/_images/sphx_glr_plot_comparison_over_sampling_001.png" alt="Drawing" width="400" />

(Fuente de la imagen: User guide de la biblioteca imbalanced-learn)


### Conjuntos de datos desbalanceados: Oversampling

- Oversampling: generar nuevas instancias de la clase minoritaria para equilibrar la cantidad de instancias de cada clase

- Random oversampling: muestreo con repetici?n instancias de la clase minoritaria



### Conjuntos de datos desbalanceados: Oversampling

- SMOTE: genero instancias "sint?ticas" a partir de los vecinos m?s cercanos.
    - Dado $x_i$, considero sus k vecinos m?s cercanos y elijo uno ($x_{zi}$) y genero una nueva instancia $x_{new} = x_i + \lambda (x_{zi} - x_i)$
    - Existen variantes de SMOTE que, por ejemplo, genera solamente a partir de instancias "en peligro" (al menos la mitad de los vecinos son de otra clase). 

<img src="https://imbalanced-learn.org/stable/_images/sphx_glr_plot_illustration_generation_sample_001.png" alt="Drawing" width="300" />

### Conjuntos de datos desbalanceados: Undersampling

- Undersampling: reducir la cantidad de instancias de la clase mayoritaria

- Random undersampling: elegir un subjconjunto aleatorio de instancias (Prototype Selection)

- Nearmiss: elegir las instancias m?s cercanas a las de la clase minoritaria utilizando el algoritmo de vecinos m?s cercanos  (Prototype Selection)

<img src="https://imbalanced-learn.org/stable/_images/sphx_glr_plot_illustration_nearmiss_001.png" alt="Drawing" width="300" />


- Mas info: imbalanced-learn User guide


### Conjuntos de datos desbalanceados: Undersampling

- ClusterCentroids: dividir las instancias de la clase mayoritaria en N clusters y utilizar los centroides como nuevos instancias (Prototype generation)

<img src="https://miro.medium.com/max/1400/1*EaLiJ5afqPgp9DUU9ykNCg.png" alt="Drawing" width="500" />


- Mas info: imbalanced-learn User guide

### Selecci?n de atributos

- Luego de que tenemos atributos "candidatos", nos gustar?a quedarnos con aquellos que "valen la pena" para la tarea de clasificaci?n que intentamos hacer. No hay una definici?n obvia de "vale la pena", pero nos interesan (para evitar ruido, y tambi?n por razones de eficiencia computacional) aquellos atributos que, en conjunto, sirvan para mejorar nuestra predicci?n. 

- El objetivo de la selecci?n de atributos es eliminar atributos que son irrelevantes o redundantes. Por ejemplo: si tenemos dos atributos con valores id?nticos, podemos eliminar uno de ellos. O si un atributo tiene siempre el mismo valor. O, por el contrario, todos sus valores son diferentes (en este caso, servir? como predictor perfecto de la clase objetivo si lo memorizamos, pero seguramente su capacidad de generalizaci?n ser? nula).


### Selecci?n de atributos - M?todos de filtrado

- Este tipo de m?todos intentan evaluar (por separado) qu? tan bueno es cada atributo. 

- Una forma muy b?sica: seleccionar los atributos que superan cierto valor de varianza. Se basa en la idea de que los atributos que valen siempre lo mismo seguramente no sean buenos predictores. Defecto: no utilizan la clase objetivo.  

- $ \chi^2$ (chi-squared): cuando tenemos dos variables categ?ricas, podemos utilizar este m?todo para estimar, a partir de los valores observadores y los esperados, si las dos variables son independientes. Cuanto mayor es el valor, mayor su correlaci?n. Por lo tanto, podemos utilizar esta medida para obtener los atributos m?s correlacionados con la clase objetivo (y que, por lo tanto, deber?an ser mejores predictores).


- Ganancia de informaci?n: la ganancia de informaci?n, o informaci?n mutua de dos variables aleatorias es una medida (proveniente de la teor?a de la informaci?n) que indica qu? tanto podemos saber de una de ellas conociendo la otra, es decir qu? tan dependientes son entre s?. Vale 0 s? y solo s? ambas variables son independientes. Por lo tanto, esta medida puede ser utilizada en forma an?loga al test $ \chi^2$ para eliminar atributos independientes de la clase objetivo.

### Selecci?n de atributos - M?todos _wrappers_

- Este tipo de m?todos de selecci?n utilizan un m?todo de aprendizaje para evaluar diferentes combinaciones de atributos y seleccionar la que obtenga mejores resultados sobre un dataset heldout separado, o v?a cross-validation. Para que funcionen, el m?todo debe ser capaz de asignar un valor de importancia a cada atributo, luego de entrenado el modelo. 

- Por ejemplo, el m?todo de **eliminaci?n recursiva de atributos** parte del conjunto inicial de atributos y, aplicando un m?todo de aprendizaje (por ejemplo, ?rboles de decisi?n) sobre un conjunto de validaci?n, elimina aquellos atributos menos importantes. Se repite el proceso hasta llegar a un cierto n?mero de atributos deseados. 

### Selecci?n de atributos - M?todos _embedded_

- A diferencia de los anteriores, la selecci?n de atributos es realizada por el mismo algoritmo de aprendizaje, durante el proceso de entrenamiento.

- Un ejemplo ya visto en el curso es el de la selecci?n de atributos que realizan los ?rboles de decisi?n.

- Otro ejemplo, que veremos m?s adelante en el curso, son los m?todos de regularizaci?n, que buscan generar modelos m?s "sencillos", para evitar el sobreajuste.

Lectura: [An Introduction to Feature Selection](https://machinelearningmastery.com/an-introduction-to-feature-selection/) - Jason Brownlee

## Fase 3: Entrenamiento

<img src="https://scikit-learn.org/stable/_images/grid_search_workflow.png" alt="Drawing" width=400/>


### Entrenamiento

- Durante el entrenamiento, utilizamos los datos del conjunto de entrenamiento, y un algoritmo de aprendizaje, para generar un clasificador.

- Una vez generado el modelo, lo evaluamos en el conjunto de evaluaci?n para obtener una medida de su performance (veremos m?s adelante las principales medidas utilizadas).

- Los algoritmos de entrenamiento tienen usualmente hiperpar?metros que deber?an ajustarse (e.g. profundidad m?xima de un ?rbol en los ?rboles decisi?n). Usualmente lo que se hace es probar diferentes valores para cada par?metro y ver cu?l obtiene mejores resultados. Como no queremos hacer esto sobre el corpus de evaluaci?n (?por qu??), utilizamos un subconjunto del corpus de entrenamiento, el corpus de validaci?n (o corpus held-out). 

### Titanic: entrenamiento


In [ ]:
from IPython.display import Image, display

# Vamos a entrenar un ?rbol de decisi?n sobre los datos de entrenamiento
from sklearn import tree
clf = tree.DecisionTreeClassifier(criterion='entropy', max_depth=3 , min_samples_leaf=5)
clf = clf.fit(X_train,y_train)

dot_code=tree.export_graphviz(clf, feature_names=['age','sex','1st_class','2nd_class','3rd_class'])

tree = graphviz.Source(dot_code)

tree.render(filename='titanic', format='png')
display(Image(filename='titanic.png'))

### Validaci?n cruzada y selecci?n de modelos

- Para utilizar mejor el corpus de entrenamiento y no tener que separar datos para validaci?n, una alternativa es realizar validaci?n cruzada. Este m?todo, adem?s, permite disminuir la varianza de los resultados (ya que se obtiene como el promedio de varias evaluaciones), aunque es m?s costoso en t?rminos de tiempo de entrenamiento.

<img src="https://scikit-learn.org/stable/_images/grid_search_cross_validation.png" alt="Drawing" width=400/>



### Validaci?n cruzada y selecci?n de modelos


- En la validaci?n cruzada, se divide el dataset de entrenamiento en k partes, y se utilizan (k-1) partes para entrenar, y la restante para evaluar el modelo. Este proceso se repite cambiando la parte elegida. 

- Se devuelve el promedio del valor de performance obtenido, y tambi?n la desviaci?n est?ndar de los resultados.

<img src="http://ethen8181.github.io/machine-learning/model_selection/img/kfolds.png" alt="Drawing" width=400/>

Fuente de las im?genes anteriores y lectura recomendada: [Cross-validation: evaluating estimator performance?](https://scikit-learn.org/stable/modules/cross_validation.html)


### Titanic: ajuste de par?metros

In [ ]:
from sklearn import metrics
import scipy.stats

# Hacemos cross validation para encontrar la mejor profundidad para el ?rbol
for md in range(10):
    clf = tree.DecisionTreeClassifier(criterion='entropy', max_depth=md+1 , min_samples_leaf=5)
    kf=sklearn.model_selection.KFold(n_splits=5)
    scores=np.zeros(5)
    score_index=0
    for train_index, test_index in kf.split(X_train):
        X_train_cv, X_test_cv= X_train.iloc[train_index], X_train.iloc[test_index]
        y_train_cv, y_test_cv= y_train.iloc[train_index], y_train.iloc[test_index]
        clf = clf.fit(X_train_cv,y_train_cv)
        y_pred=clf.predict(X_test_cv)
        scores[score_index]=metrics.accuracy_score(y_test_cv.astype(int), y_pred.astype(int))
        score_index += 1
    print ("Profundidad {0:d}, Accuracy media: {1:.3f} (+/-{2:.3f})".format(md+1, np.mean(scores), scipy.stats.sem(scores)))

## Fase 4: Evaluaci?n



- El paso final es evaluar la performance del modelo (clasificador) obtenido sobre un conjunto de datos no vistos previamente. Hasta ahora, no hemos dicho c?mo medimos esa performance

- Imaginemos, en principio, un clasificador binario. 
- Supongamos que nuestras instancias de entrenamiento tienen una cierta distribuci?n $D$, y que son independientes.

<img src="https://raw.githubusercontent.com/pln-fing-udelar/curso_aa/master/img/clasificador.png" alt="Drawing" width="400"/>



### Accuracy

- Lo m?s sencillo es estimar el acierto (accuracy) o el error.

<img src="https://raw.githubusercontent.com/pln-fing-udelar/curso_aa/master/img/accuracy.png" alt="Drawing" width=400 />

$$ acc_s = \frac{V_{azul} + V_{amarillo}}{V_{azul}+V_{amarillo} + F_{azul}+F_{amarillo}}$$  

$$ error_s = \frac{F_{azul} + F_{amarillo}}{V_{azul}+V_{amarillo} + F_{azul}+F_{amarillo}} = 1 - acc $$

### Accuracy

Ejemplo: tenemos 100 instancias ($n = 100$) donde evaluar una hip?tesis $h$, y dos clases posibles  {1,0} (consideramos a la clase 1 como la clase positiva).

Podemos construir la siguiente tabla, llamada _matriz de confusi?n_:

|  A   | h(x)=1 | h(x)=0 | Total |
|-----|----------:|----------:|-------:|
| y=1 |   $48_{TP}$    | $12_{FN}$       | 60    |
| y=0 |    $5_{FP}$     | $35_{TN}$       | 40    |
|     |    53    | 47       | 100   |

Accuracy (?cu?ntos casos predice h correctamente?) 

$$
\frac{TP + TN}{Total} = \frac{48+35}{100} = 0.83
$$


### Titanic: accuracy

In [ ]:
from sklearn import metrics
def measure_performance(X,y,clf, show_accuracy=True, show_classification_report=True, show_confusion_matrix=True):
    y_pred=clf.predict(X)   
    if show_accuracy:
        print ("Accuracy:{0:.3f}".format(metrics.accuracy_score(y,y_pred)),"\n")

    if show_classification_report:
        print("Classification report")
        print(metrics.classification_report(y,y_pred),"\n")
        
    if show_confusion_matrix:
        print ("Confusion matrix")
        print (metrics.confusion_matrix(y,y_pred),"\n")
    

In [ ]:
# Construimos un clasificador con el mejor par?metro, y entrenamos sobre todo el conjunto de entrenamiento

clf_dt=tree.DecisionTreeClassifier(criterion='entropy', max_depth=2 ,min_samples_leaf=5)
clf_dt.fit(X_train,y_train)
measure_performance(X_test,y_test,clf_dt, show_classification_report=False, show_confusion_matrix=False)

### Evaluaci?n de hip?tesis

- ?Es el error en la muestra un buen estimador del error "real"? ?Cu?l es la probabilidad de error de este estimador? 

Primero, definamos el error cometido por una cierta hip?tesis en una muestra: 

$$
error_s(h) \equiv \frac{1}{n}\Sigma \delta(y,h(x))
$$

donde $\delta(x,y)$ es 1 si los valores de $x$ y $y$ son distintos, y 0 en otro caso.

El error real es simplemente 

$$
error_D(h) \equiv P_{x \in D} (y \neq h(x))
$$

### Evaluaci?n de hip?tesis

Si 
- Nuestras hip?tesis toman valores discretos
- Nuestras instancias son independientes entre s?, y de la hip?tesis
- $n \geq 30$
- $error_s(h) = r/n$
- $error_s(h)$ no est? demasiado cercano a 0 o 1 (Regla: $n\cdot error_s(h)(1-error_s(h))\geq 5)$

Entonces 

1. Un estimador no sesgado de $error_D(h)$ es $error_s(h)$
2. El 95% de las veces, $error_d(h)$ cae en el intervalo (llamado _intervalo de confianza_):

$$
error_s(h) \pm 1.96 \sqrt{\frac{error_s(h)(1 - error_s(h))}{n}}
$$


### Evaluaci?n de hip?tesis

Ejemplo: supongamos que nuestra hip?tesis se equivoca en 12 de 40 instancias. Entonces, el intervalo de confianza 95% es: 

$$
0.3 \pm 1.96 \sqrt {\frac{0.3 \times 0.7)}{40}} = 0.3 \pm 0.14
$$

Si evaluamos sobre 4000 instancias, y nos equivocamos 1200 veces, el intervalo es: 

$$
0.3 \pm 1.96 \sqrt {\frac{0.3 \times 0.7)}{4000}} = 0.3 \pm 0.0014
$$



### Evaluaci?n de hip?tesis

?De d?nde salen esos valores? 

- Evaluar una hip?tesis n veces y ver cu?ntas se equivoca es como tirar n veces una moneda y ver cu?ntas caras salen
- La distribuci?n es una binomial
- La esperanza de una binomial es $np$, siendo $n$ el tama?o de la muestra y $p$ el valor a estimar
- La desviaci?n est?ndar de una binomial es $\sigma = \sqrt{np(1-p)}$

Para una distribuci?n binomial

$error_s(h)=r/n$, por lo que $E[error_s(h)]=p$ 

por lo que el error en la muestra es un estimador no sesgado de p. 

- Para calcular la desviaci?n est?ndar, sustituimos $p$ por nuestra estimaci?n. 


$\sigma_{error_s(h)} = \sigma_{r/n} = \sqrt{Var(r/n)} = \sqrt{\frac{1}{n^2} Var(r)}  = \frac{\sigma_r}{n} = \sqrt{\frac{p(1-p)}{n}} \approx \sqrt{\frac{error_s(h)(1-error_s(h))}{n}} $

- Para calcular el intervalo de confianza, aproximamos la binomial con una distribuci?n normal

(Quien est? interesado en los detalles, puede consultar la secci?n 5.2 y 5.3 del libro de Mitchell)


### Precisi?n y Recuperaci?n

- El problema con el acierto y el error es que no tienen en cuenta el
comportamiento en las distintas clases.
- Si el 99% de las instancias son azules, la funci?n constante azul tiene
un acierto de 99%.
- Se buscan alternativas para medir por clase: precisi?n, recuperaci?n (_recall_)


$$ precision = \frac{V_p} {V_p + F_p} $$

$$ recall = \frac{V_p}  {V_p+F_n} $$

 

$V_p$ indica verdaderos positivos, es decir aquellos ejemplos que fueron clasificados correctamente

$F_p$ indica falsos positivos, es decir aquellos ejemplos que fueron clasificados como positivos, pero eran negativos

$V_n$ indica verdaderos negativos

$F_n$ indica falsos negativos 



### Precisi?n y Recuperaci?n

Ejemplo: tenemos 100 instancias ($n = 100$) donde evaluar una hip?tesis $h$, y dos clases posibles  {1,0} (consideramos a la clase 1 como la clase positiva).

Podemos construir la siguiente tabla, llamada _matriz de confusi?n_:

|  A   | h(x)=1 | h(x)=0 | Total |
|-----|----------:|----------:|-------:|
| y=1 |   $48_{TP}$    | $12_{FN}$       | 60    |
| y=0 |    $5_{FP}$     | $35_{TN}$       | 40    |
|     |    53    | 47       | 100   |

Precision (?De los que h predijo positivos, cu?ntos lo eran?)

$$
\frac{TP }{h(x)=1} = \frac{48}{53} = 0.91
$$


### Precisi?n y Recuperaci?n

Ejemplo: tenemos 100 instancias ($n = 100$) donde evaluar una hip?tesis $h$, y dos clases posibles  {1,0} (consideramos a la clase 1 como la clase positiva).

Podemos construir la siguiente tabla, llamada _matriz de confusi?n_:

|  A   | h(x)=1 | h(x)=0 | Total |
|-----|----------:|----------:|-------:|
| y=1 |   $48_{TP}$    | $12_{FN}$       | 60    |
| y=0 |    $5_{FP}$     | $35_{TN}$       | 40    |
|     |    53    | 47       | 100   |

Recall (?Cu?ntos de los positivos pudo h predecir correctamente?)

$$
\frac{TP }{y=1} = \frac{48}{60} = 0.80
$$


### Precisi?n y Recuperaci?n

?Qu? pasa si consideramos a 0 como la clase positiva? Es equivalente a repetir las medidas anteriores, pero con los n?meros cambiados en filas y columnas.


|  A   | h(x)=1 | h(x)=0 | Total |
|-----|----------:|----------:|-------:|
| y=1 |   $35_{TP}$    | $5_{FN}$       | 40    |
| y=0 |    $12_{FP}$     | $48_{TN}$       | 60    |
|     |    47    | 53       | 100   |

Accuracy (?cu?ntos casos predice h correctamente?): $\frac{TP + TN}{Total} = \frac{35+48}{100} = 0.83$     

Precision (?De los que h predijo positivos, cu?ntos lo eran?): $\frac{TP }{h(x)=1} = \frac{35}{47} = 0.74$ 

Recall (?Cu?ntos de los positivos pudo h predecir correctamente?): $\frac{TP }{h(x)=1} = \frac{35}{40} = 0.88$

La precisi?n y el recall son siempre respecto a una clase positiva, el accuracy es una medida general. 


### Medida-F
- Combinando precisi?n y recuperaci?n se obtiene la medida-F (donde $\beta$ indica cu?nta m?s importancia se le da al recall respecto a la precisi?n) : 

$$F_\beta = \frac{(1+ \beta^2) \cdot  precision \cdot recall}{\beta^2 \cdot precision + recall}$$

- En el caso de $F_1$, la formula queda reducida a:

$$F_1 = \frac{2 \cdot  precision \cdot recall}{ precision + recall}$$

- La medida-F es la media arm?nica entre precisi?n y recall, e intenta combinar ambas en un s?lo n?mero. 

Para una interesante discusi?n sobre por qu? se utiliza la media arm?nica y no la media aritm?tica, sugerimos este art?culo: [The truth of the F-measure](https://www.cs.odu.edu/~mukka/cs795sum09dm/Lecturenotes/Day3/F-measure-YS-26Oct07.pdf) - Yutaka Sasaki

### Medida-F


Ejercicio: complete la siguiente tabla, para una clasificaci?n sobre 100.000 instancias


| $V_p$ | $F_p$ | $F_n$ | $V_n$ | Prec | Recall | $F_1$ | Accuracy |
| :---: |:---:  | :---: | :---: | :---: | :---: | :---: | :---: |
| 25    |   0   | 125   | 99850 | 1.00 | 0.17  | 0.29 | 0.999    |
| 50    |  100   | 100   | 99750 |0.33 | 0.33 |0.33 |0.999  | 
| 75    |   150   | 75   | 99700 | 0.33 | 0.50|0.40|0.998 |  
| 100    |   50  | 50  | 99800 | 0.67 | 0.67|0.67|0.999  |
| 150    |   100   | 0   | 99750 | 0.60 |1.00| 0.75| 0.999 |

- ?Qu? sucede con la accuracy? ?Y con los otros valores? ?Cu?l de los clasificadores eligir?a?


### Matriz de confusi?n

Las matrices de confusi?n son muy ?tiles para visualizar c?mo se comporta una hip?tesis en  problemas con varias clases. En el siguiente ejemplo (tomado de [aqu?](https://www.blackhc.net/blog/2019/mnist-by-zip/)), se muestra una matriz de confusi?n para un clasificador de d?gitos escritos a mano (que no funciona muy bien, dicho sea de paso...). En este caso, los valores fueron normalizados, esto es, var?an entre 0 y 1 (?imagina para qu??)

<img src="https://www.blackhc.net/blog/2019/mnist-by-zip/confusion_matrix_45.png" alt="Drawing" width=400/>



### Titanic: Precision, Recall, medida-F


In [ ]:
# Construimos un clasificador con el mejor par?metro, y entrenamos sobre todo el conjunto de entrenamiento
measure_performance(X_test,y_test,clf_dt)

### Problemas multiclase

- ?Qu? sucede cuando se tiene un problema multiclase (es decir, hay m?s de dos categor?as)?
- Las medidas anteriores siguen valiendo, si consideramos como "positivos" a las instancias que pertenecen a una clase, y "negativos" al resto (one-versus-all).



### Problemas multiclase

Supongamos que tenemos tres clases a,b,c, y la siguiente matriz de confusi?n: 

|  A   | h(x)=a | h(x)=b | h(x)=c | Total |
|-----|----------:|----------:|----:|-------:|
| y=a |   $20$    | $10$       | $10$ |   30    |
| y=b |    $3$     | $21$       |$6$ |  30    |
| y=c |   $0$    | $1$       | $30$ |  40    |
|     |    23   | 41  | 46     | 100   |

Accuracy (general) : $\frac{20+21+30}{100} = 0.71 $


### Problemas multiclase

Supongamos que tenemos tres clases a,b,c, y la siguiente matriz de confusi?n: 

|  A   | h(x)=a | h(x)=b | h(x)=c | Total |
|-----|----------:|----------:|----:|-------:|
| y=a |   $20_{TP}$    | $10_{FN}$       | $10_{FN}$ |   30    |
| y=b |    $3_{FP}$     | $21_{TN}$       |$6_{TN}$ |  30    |
| y=c |   $0_{FP}$    | $1_{TN}$       | $30_{TN}$ |  40    |
|     |    23   | 41  | 46     | 100   |

$Precision_A  : \frac{TP}{h(x)=a} = \frac{20}{23} = 0.87 $


$Recall_A$  : $\frac{TP}{y=a} = \frac{20}{30} = 0.67 $


(Ejercicio, verifique $Pr_b=0.51$, $R_b=0.7$, $Pr_c=0.65$, $R_c=0.75$)



### Problemas multiclase

- Esto nos da una medida por cada clase. Existen diferentes formas de resumir esa informaci?n:

    Se calcula la medida por clase, y luego se promedia los valores obtenidos (macro average)
    
    Se calcula la medida por clase teniendo en cuenta el aporte de instancias cada clase (micro average)
    
En el ejemplo anterior: 
- La macro-average de la precision ser? $ \frac{(Pr_a + Pr_b + Pr_c)}{3} = \frac{0.87 + 0.51 + 0.65}{3} = 0.67 $
- La micro-average ser? $ \frac{20+21+30}{23+41+46} = 0.71 $ (en un problema multiclase, la micro es igual a la accuracy!)



- Qu? medida es m?s "?til", como toda medida, **depende de lo que queremos evaluar**:
    - La micro-average da m?s peso a las clases grandes en el an?lisis general
    - La macro-average permite evaluar mejor que tan "equilibrado" es el comportamiento de mi clasificador 
    - En lo posible, reportar ambas y analizar seg?n mi problema
    
Lectura recomendada: [Should I make decisions based on micro-averaged or macro-averaged evaluation measures?](https://stats.stackexchange.com/questions/156923/should-i-make-decisions-based-on-micro-averaged-or-macro-averaged-evaluation-mea)


### Problemas multietiqueta


- En un problema multietiqueta, hay m?s de una clase asociada a cada etiqueta, como en el siguiente ejemplo

| Instancia | Clase | Predicci?n |
| --- |:---:  | :---: | 
| 1   | A,B   | B,C | 
| 2    | A,B,C   | A,C,D |
| 3    | A,B   | A,B |

- Se pueden calcular las medidas utilizando solamente los ejemplos de cada clase, sin importar el resto. Esto permite utilizar las medidas mencionadas anteriormente

- Se pueden aplicar otras medidas, como promediar el ?ndice de Jaccard $IJ(A,B) = |A \cap B| / |A \cup B|$

- En el ejemplo, $IJ(1)=1/3$, $IJ(2)=1/2$, $IJ(3)=1$, y por lo tanto el ?ndice de Jaccard promedio ser? $0.61$


### Variables ordinales

?Qu? pasa si queremos predecir variables ordinales (e.g. cantidad de c?rners de un partido)? 

<img src="https://miro.medium.com/max/860/1*ya6S6ACUoPyayveSW2iC_w.png" alt="Drawing" width=400 />

Posible estad?stica: accuracy_n (predecir el valor d, o hasta d+n)


### L?nea base  y l?nea m?xima

- ?C?mo sabemos si el resultado que obtuvimos es "bueno", o "razonable"?

- El resultado depende del problema. 

- Siempre es bueno tener una l?nea base: una soluci?n anterior sencilla, o un clasificador que elige siempre la clase m?s probable o seg?n la distribuci?n del conjunto de entrenamiento

- Tambi?n es ?til (si es posible) tener una l?nea de tope, sobre todo en problemas donde no hay antecedentes. T?picamente, se pide a humanos que act?en como clasificadores y se eval?a su performance. 
